In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

In [ ]:
device = torch.device("cuda")

In [ ]:
import torch

# Check GPU is available
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

sam2_model = build_sam2(
    config_file="configs/sam2.1/sam2.1_hiera_l.yaml",
    ckpt_path="checkpoints/sam2.1_hiera_large.pt",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

predictor = SAM2ImagePredictor(sam2_model)
print("Model loaded successfully!")

In [ ]:
import json
import numpy as np
from PIL import Image
from shapely import wkt
import os

DATA_DIR = "/home/sagemaker-user/42028-DLCNN/data/xbd/train"

def load_sample(disaster_name, sample_id):
    """Load a pre/post image pair and their building polygons + damage labels."""
    
    base = f"{disaster_name}_{sample_id:08d}"
    
    # Load images
    pre_img  = np.array(Image.open(f"{DATA_DIR}/images/{base}_pre_disaster.png").convert("RGB"))
    post_img = np.array(Image.open(f"{DATA_DIR}/images/{base}_post_disaster.png").convert("RGB"))
    
    # Load post label (has damage classifications)
    with open(f"{DATA_DIR}/labels/{base}_post_disaster.json") as f:
        label = json.load(f)
    
    buildings = []
    for feature in label["features"]["xy"]:
        poly = wkt.loads(feature["wkt"])
        damage = feature["properties"]["subtype"]
        coords = np.array(poly.exterior.coords, dtype=np.int32)
        buildings.append({"polygon": coords, "damage": damage})
    
    return pre_img, post_img, buildings

# Test it
pre, post, buildings = load_sample("guatemala-volcano", 0)
print(f"Pre image shape:  {pre.shape}")
print(f"Post image shape: {post.shape}")
print(f"Buildings found:  {len(buildings)}")
print(f"First building:   {buildings[0]}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

DAMAGE_COLORS = {
    "no-damage":    (0, 255, 0),    # green
    "minor-damage": (255, 255, 0),  # yellow
    "major-damage": (255, 165, 0),  # orange
    "destroyed":    (255, 0, 0),    # red
}

def visualize_sample(post_img, buildings):
    vis = post_img.copy()
    for b in buildings:
        color = DAMAGE_COLORS.get(b["damage"], (255, 255, 255))
        cv2.polylines(vis, [b["polygon"]], isClosed=True, color=color, thickness=2)
    
    plt.figure(figsize=(12, 12))
    plt.imshow(vis)
    plt.title("Post-disaster buildings (colored by damage level)")
    plt.axis("off")
    
    # legend
    patches = [mpatches.Patch(color=np.array(c)/255, label=k) for k, c in DAMAGE_COLORS.items()]
    plt.legend(handles=patches, loc="upper right")
    plt.show()

visualize_sample(post, buildings)

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import torch

def get_bbox(polygon):
    """Get bounding box [x0, y0, x1, y1] from polygon coords."""
    x0, y0 = polygon[:, 0].min(), polygon[:, 1].min()
    x1, y1 = polygon[:, 0].max(), polygon[:, 1].max()
    return np.array([x0, y0, x1, y1])

# Set image for SAM2
predictor.set_image(post)

# Run SAM2 on each building
results = []
for b in buildings:
    bbox = get_bbox(b["polygon"])
    
    masks, scores, _ = predictor.predict(
        box=bbox,
        multimask_output=False  # one mask per building
    )
    
    results.append({
        "mask":   masks[0],       # (1024, 1024) boolean array
        "score":  scores[0],
        "damage": b["damage"],
        "bbox":   bbox
    })

print(f"Segmented {len(results)} buildings")
print(f"Example — damage: {results[0]['damage']}, score: {results[0]['score']:.3f}, mask sum: {results[0]['mask'].sum()}")

In [ ]:
def visualize_masks(post_img, results):
    vis = post_img.copy().astype(np.float32)
    
    for r in results:
        color = np.array(DAMAGE_COLORS[r["damage"]], dtype=np.float32)
        mask = r["mask"].astype(bool)
        vis[mask] = vis[mask] * 0.4 + color * 0.6
    
    vis = vis.astype(np.uint8)
    
    plt.figure(figsize=(12, 12))
    plt.imshow(vis)
    plt.title("SAM2 masks colored by damage level")
    plt.axis("off")
    patches = [mpatches.Patch(color=np.array(c)/255, label=k) for k, c in DAMAGE_COLORS.items()]
    plt.legend(handles=patches, loc="upper right")
    plt.show()

visualize_masks(post, results)

In [ ]:
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

CROP_SIZE = 128

transform = T.Compose([
    T.Resize((CROP_SIZE, CROP_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

def extract_crop(image, bbox):
    """Crop a building region from an image using its bounding box."""
    x0, y0, x1, y1 = bbox.astype(int)
    pad = 10
    x0, y0 = max(0, x0 - pad), max(0, y0 - pad)
    x1, y1 = min(image.shape[1], x1 + pad), min(image.shape[0], y1 + pad)
    crop = image[y0:y1, x0:x1]
    return transform(Image.fromarray(crop))

# Test on first building
r = results[0]
pre_crop  = extract_crop(pre, r["bbox"])
post_crop = extract_crop(post, r["bbox"])
print(f"Pre crop shape:  {pre_crop.shape}")
print(f"Post crop shape: {post_crop.shape}")
print(f"Damage label:    {r['damage']}")

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

DAMAGE_CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]
NUM_CLASSES = len(DAMAGE_CLASSES)

class DamageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # shared ResNet18 backbone for both pre and post crops
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        # remove final classification layer, keep feature extractor
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])  # output: (batch, 512, 1, 1)
        
        # classifier takes concatenated pre+post features
        self.classifier = nn.Sequential(
            nn.Linear(512 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, NUM_CLASSES)
        )
    
    def forward(self, pre, post):
        pre_feat  = self.encoder(pre).flatten(1)   # (batch, 512)
        post_feat = self.encoder(post).flatten(1)  # (batch, 512)
        combined  = torch.cat([pre_feat, post_feat], dim=1)  # (batch, 1024)
        return self.classifier(combined)

# test forward pass
model = DamageClassifier().to(device)
pre_batch  = pre_crop.unsqueeze(0).to(device)
post_batch = post_crop.unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(pre_batch, post_batch)

print(f"Output shape: {logits.shape}")
print(f"Predicted class: {DAMAGE_CLASSES[logits.argmax().item()]}")
print("Model ready!")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import glob

class XBDDataset(Dataset):
    def __init__(self, data_dir, predictor):
        self.data_dir = data_dir
        self.predictor = predictor
        self.samples = []  # list of (pre_img, post_img, bbox, damage_label)
        self._build_samples()
    
    def _build_samples(self):
        label_files = glob.glob(f"{self.data_dir}/labels/*_post_disaster.json")
        print(f"Found {len(label_files)} post-disaster label files")
        
        for label_path in label_files:
            base = os.path.basename(label_path).replace("_post_disaster.json", "")
            pre_path  = f"{self.data_dir}/images/{base}_pre_disaster.png"
            post_path = f"{self.data_dir}/images/{base}_post_disaster.png"
            
            if not os.path.exists(pre_path) or not os.path.exists(post_path):
                continue
            
            with open(label_path) as f:
                label = json.load(f)
            
            for feature in label["features"]["xy"]:
                damage = feature["properties"]["subtype"]
                if damage not in DAMAGE_CLASSES:
                    continue
                poly = wkt.loads(feature["wkt"])
                coords = np.array(poly.exterior.coords, dtype=np.int32)
                bbox = get_bbox(coords)
                # skip tiny buildings
                if (bbox[2]-bbox[0]) < 5 or (bbox[3]-bbox[1]) < 5:
                    continue
                self.samples.append((pre_path, post_path, bbox, damage))
        
        print(f"Total buildings: {len(self.samples)}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        pre_path, post_path, bbox, damage = self.samples[idx]
        pre_img  = np.array(Image.open(pre_path).convert("RGB"))
        post_img = np.array(Image.open(post_path).convert("RGB"))
        pre_crop  = extract_crop(pre_img, bbox)
        post_crop = extract_crop(post_img, bbox)
        label = DAMAGE_CLASSES.index(damage)
        return pre_crop, post_crop, label

# build dataset
dataset = XBDDataset(DATA_DIR, predictor)

# split 80/20 train/val
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=4)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import time

import glob

def get_class_weights_fast(data_dir):
    counts = [0] * NUM_CLASSES
    label_files = glob.glob(f"{data_dir}/labels/*_post_disaster.json")
    
    for label_path in label_files:
        with open(label_path) as f:
            label = json.load(f)
        for feature in label["features"]["xy"]:
            damage = feature["properties"]["subtype"]
            if damage in DAMAGE_CLASSES:
                counts[DAMAGE_CLASSES.index(damage)] += 1
    
    total = sum(counts)
    weights = [total / (NUM_CLASSES * c) for c in counts]
    print(f"Class counts:  {dict(zip(DAMAGE_CLASSES, counts))}")
    print(f"Class weights: {dict(zip(DAMAGE_CLASSES, [round(w,2) for w in weights]))}")
    return torch.tensor(weights, dtype=torch.float).to(device)

print("Computing class weights...")
weights = get_class_weights_fast(DATA_DIR)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = Adam(model.parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=3, gamma=0.5)

def train_epoch(model, loader, optimizer, criterion, epoch):
    model.train()
    total_loss, correct, total = 0, 0, 0
    start = time.time()
    num_batches = len(loader)

    for batch_idx, (pre, post, labels) in enumerate(loader):
        pre, post, labels = pre.to(device), post.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(pre, post)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)

        # print every 50 batches
        if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == num_batches:
            elapsed = time.time() - start
            batches_done = batch_idx + 1
            eta = (elapsed / batches_done) * (num_batches - batches_done)
            print(f"  Epoch {epoch+1} | batch {batches_done}/{num_batches} | "
                  f"loss: {loss.item():.4f} | "
                  f"acc: {correct/total:.4f} | "
                  f"elapsed: {elapsed:.0f}s | ETA: {eta:.0f}s")

    return total_loss / num_batches, correct / total

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    start = time.time()
    num_batches = len(loader)

    with torch.no_grad():
        for batch_idx, (pre, post, labels) in enumerate(loader):
            pre, post, labels = pre.to(device), post.to(device), labels.to(device)
            logits = model(pre, post)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(labels)

            if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == num_batches:
                elapsed = time.time() - start
                print(f"  Val | batch {batch_idx+1}/{num_batches} | "
                      f"loss: {loss.item():.4f} | "
                      f"acc: {correct/total:.4f} | "
                      f"elapsed: {elapsed:.0f}s")

    return total_loss / num_batches, correct / total

# training loop
NUM_EPOCHS = 10
best_val_acc = 0

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} starting...")
    print(f"{'='*60}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, epoch)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)
    scheduler.step()
    epoch_time = time.time() - epoch_start

    print(f"\nEpoch {epoch+1} summary | "
          f"train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"val loss: {val_loss:.4f} acc: {val_acc:.4f} | "
          f"time: {epoch_time:.0f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/home/sagemaker-user/42028-DLCNN/checkpoints/best_model.pt")
        print(f"  -> Saved best model (val acc: {val_acc:.4f})")